<a href="https://colab.research.google.com/github/AryaLolusare2712/Build-Your-Own-ChatBot/blob/feature%2Fchatbot-enhancements/Easy_ChatBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Build Your Own AI Chatbot with Python and Gradio

In this tutorial, we will build a simple AI chatbot using Python, Gradio, and the Qwen language model.

The chatbot will:

- Run the Qwen model locally in Google Colab
- Provide a web interface using Gradio
- Remember messages from the current conversation
- Stream responses while they are being generated
- Handle errors gracefully

> **Note:** This chatbot uses a local Qwen model, so no external LLM API key is required.

This tutorial explains each major component step by step so that beginners can understand and modify the chatbot.

In [1]:
!pip install gradio==6.20.0 transformers==5.13.1 accelerate==1.14.0

## 2. Import the required libraries

We use the following libraries to build the chatbot:

- **Gradio** — creates the chatbot web interface.
- **PyTorch** — provides the underlying machine-learning framework.
- **Transformers** — loads and runs the Qwen language model.
- **Threading** — allows the model to generate text while the response is streamed to the interface.

In [2]:
import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
from threading import Thread

## 3. Load the Qwen language model

We use `Qwen/Qwen2.5-1.5B-Instruct`, a small instruction-following language model.

The model runs locally in the Google Colab environment. The first time this cell is executed, the model files will be downloaded from Hugging Face.

Because the model runs locally, this chatbot does not require an external LLM API key.

In [8]:
# ============================================================
# 1. Load the Qwen model
# ============================================================

# We use a local Qwen model.
# No external LLM API key is required.

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")


Loading tokenizer...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully!


## 4. Handle conversation messages

Gradio provides the conversation history to our chatbot.

The helper function below converts different message formats into plain text before sending them to the Qwen model.

In [9]:
# ============================================================
# 2. Convert Gradio message content into plain text
# ============================================================

def get_message_text(content):
    """
    Convert Gradio's message content into plain text.

    Gradio 6 may provide message content as:
    - a string
    - a dictionary
    - a list of content blocks
    """

    # Simple text message
    if isinstance(content, str):
        return content

    # Dictionary-based content
    if isinstance(content, dict):
        return content.get("text", "")

    # Structured content blocks
    if isinstance(content, list):
        text_parts = []

        for block in content:
            if isinstance(block, str):
                text_parts.append(block)

            elif isinstance(block, dict):
                if block.get("type") == "text":
                    text_parts.append(block.get("text", ""))

        return "".join(text_parts)

    return ""


## 5. Generate chatbot responses

The `respond()` function is the main part of the chatbot.

It:

1. Starts with a system instruction.
2. Adds the previous conversation history.
3. Adds the user's latest message.
4. Limits the amount of conversation history.
5. Converts the messages into Qwen's chat format.
6. Generates a response.
7. Streams the response back to Gradio.
8. Handles errors without crashing the interface.

In [10]:
# ============================================================
# 3. Generate chatbot response
# ============================================================

def respond(message, history):
    """
    Generate a response from the Qwen model.

    Parameters:
        message: The user's latest message.
        history: Previous conversation messages from Gradio.

    Yields:
        Partial response text while the model is generating.
    """

    try:
        # Start the conversation with a system message.
        chat_memory = [
            {
                "role": "system",
                "content": (
                    "You are a friendly and helpful chatbot. "
                    "Use the conversation history to answer questions "
                    "consistently, but do not claim to have permanent "
                    "memory outside the current chat."
                )
            }
        ]

        # ----------------------------------------------------
        # Add previous conversation history
        # ----------------------------------------------------

        if history:

            for previous_message in history:

                role = previous_message.get("role")

                content = get_message_text(
                    previous_message.get("content")
                )

                # Only add valid user/assistant messages.
                if role in ["user", "assistant"] and content:
                    chat_memory.append(
                        {
                            "role": role,
                            "content": content
                        }
                    )

        # ----------------------------------------------------
        # Add the latest user message
        # ----------------------------------------------------

        chat_memory.append(
            {
                "role": "user",
                "content": message
            }
        )

        # ----------------------------------------------------
        # Limit conversation length
        # ----------------------------------------------------
        #
        # Keeping the conversation short prevents the prompt
        # from becoming unnecessarily large.
        #

        if len(chat_memory) > 10:

            chat_memory = (
                [chat_memory[0]]
                + chat_memory[-8:]
            )

        # ----------------------------------------------------
        # Convert messages into Qwen's chat format
        # ----------------------------------------------------

        text = tokenizer.apply_chat_template(
            chat_memory,
            tokenize=False,
            add_generation_prompt=True
        )

        # ----------------------------------------------------
        # Tokenize the prompt
        # ----------------------------------------------------

        model_inputs = tokenizer(
            [text],
            return_tensors="pt"
        ).to(model.device)

        # ----------------------------------------------------
        # Create a streamer
        # ----------------------------------------------------

        streamer = TextIteratorStreamer(
            tokenizer,
            skip_prompt=True,
            skip_special_tokens=True
        )

        # ----------------------------------------------------
        # Generation settings
        # ----------------------------------------------------

        generation_kwargs = {
            **model_inputs,
            "streamer": streamer,
            "max_new_tokens": 512,
            "temperature": 0.8
        }

        # ----------------------------------------------------
        # Run model generation in a separate thread
        # ----------------------------------------------------

        thread = Thread(
            target=model.generate,
            kwargs=generation_kwargs
        )

        thread.start()

        # ----------------------------------------------------
        # Stream response to Gradio
        # ----------------------------------------------------

        partial_text = ""

        for new_text in streamer:

            partial_text += new_text

            yield partial_text

        # Make sure the generation thread has finished.
        thread.join()

    except Exception as error:

        # Print the actual error in Colab.
        print("\n========== CHATBOT ERROR ==========")
        print(error)
        print("===================================\n")

        # Show a user-friendly message.
        yield (
            "Sorry, something went wrong while generating "
            "the response. Please try again."
        )


## 6. Create the Gradio interface

Gradio provides the user interface for our chatbot.

`ChatInterface` connects the user's message to the `respond()` function and provides controls for interacting with the conversation.

In [11]:
# ============================================================
# 4. Create Gradio ChatInterface
# ============================================================

chatbot = gr.ChatInterface(
    fn=respond,

    title="🤖 My Local AI Chatbot",

    description=(
        "Chat with a Qwen language model running locally "
        "in Google Colab. No external LLM API key is required."
    ),

    textbox=gr.Textbox(
        placeholder="Ask me anything...",
        container=True
    )
)


## 7. Launch the chatbot

The following cell starts the Gradio application.

When running in Google Colab, `share=True` creates a temporary public link that can be used to access the chatbot.

> **Security note:** Do not enter sensitive or private information into a publicly shared demo.

In [13]:
# ============================================================
# 5. Launch the chatbot
# ============================================================

# share=True creates a temporary public Gradio URL.
# This is useful when running the notebook in Google Colab.

chatbot.launch(share=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fabfcb658d545912f6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 8. Troubleshooting

### The model takes a long time to load

The Qwen model needs to download its files the first time the notebook is run. Loading time depends on the available hardware and network connection.

### The chatbot runs out of memory

Try reducing:

```python
max_new_tokens=512
```
to:
```python
max_new_tokens=256
```